# VERITAS-T — Full-Scale Kaggle Run (K=50, Two Models, Checkpointed)

**Before running:**
1. Upload `medical_labels.csv`, `medical_outputs.csv`, `legal_train.tsv`, `legal_test.tsv`, `finance_data.csv` as a Kaggle Dataset.
2. Attach the dataset to this notebook; update `DATASET_BASE` in Cell 2 if your dataset slug differs from `veritas-datasets`.
3. Attach a **GPU accelerator** (T4 x2) under notebook settings.
4. Add your HuggingFace token as a Kaggle **Secret** named `HF_TOKEN` (Add-ons -> Secrets) — required for `meta-llama/Meta-Llama-3-8B-Instruct`.

**What this does:**
- Runs the full VERITAS-T pipeline at K=50 with real per-domain sample sizes (capped at 1500/domain by default).
- Extracts REAL filing dates from FinanceBench context for temporal validation (no longer simulated for Finance).
- Runs TWO models in sequence: Llama-3-8B-Instruct, then Qwen2.5-7B-Instruct.
- Checkpoints every 30 minutes (or every 200 claims) to `/kaggle/working/results/checkpoint_<model>.csv` — safe to stop and resume.
- After EACH model finishes, writes a complete, independent result bundle (CSV + manifest) and prints a summary — you can stop here, download, and share before deciding whether to run the second model.

In [1]:
!pip install -q transformers torch sentence-transformers openpyxl tqdm accelerate

## Main pipeline — runs both models in sequence with checkpointing

In [2]:
import os, sys, json, time, math, logging, re
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm import tqdm

from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill
from sentence_transformers import SentenceTransformer
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# ── Kaggle secrets (HF token) ───────────────────────────────────────────────
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
    os.environ["HF_TOKEN"] = HF_TOKEN
    print("HF_TOKEN loaded from Kaggle secrets.")
except Exception as e:
    HF_TOKEN = os.environ.get("HF_TOKEN", None)
    print(f"[WARN] Could not load HF_TOKEN from Kaggle secrets ({e}). "
          f"Gated models (Llama-3) will fail without it.")

# ── CONFIG ───────────────────────────────────────────────────────────────────
DATASET_BASE = Path("/kaggle/input/datasets/samuelstephen77/veritas-datasets")   # <-- change to your dataset slug
OUTPUT_DIR   = Path("/kaggle/working/results")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MAX_PER_DOMAIN = 1500     # cap per domain; set None for true full run
K              = 50       # paper-spec samples per claim
GEN_MAX_NEW_TOKENS = 64
GEN_TEMPERATURE    = 0.7
EMBED_MODEL        = "all-MiniLM-L6-v2"

# Models to run, IN ORDER. Each produces its own complete result bundle.
MODELS_TO_RUN = [
    {"id": "Qwen/Qwen2.5-7B-Instruct",            "nickname": "qwen2.5-7b", "needs_token": False},
    {"id": "meta-llama/Meta-Llama-3-8B-Instruct", "nickname": "llama3-8b", "needs_token": True},
    
]

CHECKPOINT_EVERY_SECONDS = 1800   # 30 minutes
CHECKPOINT_EVERY_N_CLAIMS = 200   # whichever comes first

CONFIG = {
    "medical_labels":  DATASET_BASE / "medical_labels.csv",
    "medical_outputs": DATASET_BASE / "medical_outputs.csv",
    "legal_train":     DATASET_BASE / "legal_train.tsv",
    "legal_test":       DATASET_BASE / "legal_test.tsv",
    "finance_data":    DATASET_BASE / "finance_data.csv",
    "K": K, "alpha": 0.60, "beta": 0.30, "seed": 42, "bootstrap_B": 1000,
    "C_medical": 1.0, "C_legal": 0.8, "C_finance": 0.9,
    "S_high": 0.8, "S_neutral": 0.5, "S_low": 0.2,
    "lambda_t": 0.02,
    "t_medical_range": (1, 6), "t_legal_range": (6, 36), "t_finance_range": (1, 12),
}
np.random.seed(CONFIG["seed"])
logging.basicConfig(level=logging.INFO, format="[%(levelname)s] %(message)s")
log = logging.getLogger("VERITAS-T")

LEGAL_HIGH_KEYWORDS   = ["statute","section","court","case","ruling","penalty","liability",
                          "judgment","convicted","breach","regulation","compliance","infringement"]
FINANCE_HIGH_KEYWORDS = ["revenue","earnings","eps","profit","loss","debt","liability",
                          "dividend","share price","guidance","forecast","operating income",
                          "net income","cash flow","equity","assets","turnover"]
MONTHS = "January|February|March|April|May|June|July|August|September|October|November|December"
DATE_RE = re.compile(rf"({MONTHS})\s+\d{{1,2}},?\s+(\d{{4}})")

# ── CORE FORMULA ─────────────────────────────────────────────────────────────
def per_claim_penalty(F, C, S, M):
    return float(C * S * (1.0 - F) * (1.0 - M))

def veritas_score(claims):
    if not claims: return 0.0
    total_risk = sum(per_claim_penalty(c["F"], c["C"], c["S"], c["M"]) for c in claims)
    total_weight = sum(c["C"] for c in claims)
    return float(np.clip(1.0 - (total_risk / max(total_weight, 1e-9)), 0.0, 1.0))

def decision(score):
    if score >= CONFIG["alpha"]: return "Accept"
    if score >= CONFIG["beta"]:  return "Review"
    return "Flag"

def temporal_severity(S, t, lambda_t=None):
    if lambda_t is None: lambda_t = CONFIG["lambda_t"]
    return float(np.clip(S * math.exp(-lambda_t * t), 0.0, 1.0))

def simulate_t(domain, n, seed=42):
    rng = np.random.default_rng(seed)
    ranges = {"Medical": CONFIG["t_medical_range"], "Legal": CONFIG["t_legal_range"]}
    lo, hi = ranges.get(domain, (1, 12))
    return rng.uniform(lo, hi, size=n)

def extract_finance_t(context_text, reference_date=datetime(2024, 6, 30)):
    matches = DATE_RE.findall(str(context_text))
    if not matches: return None
    years = [int(y) for _, y in matches]
    most_recent_year = max(years)
    months_map = {m: i+1 for i, m in enumerate(MONTHS.split("|"))}
    candidate_months = [months_map[m] for m, y in matches if int(y) == most_recent_year]
    month = max(candidate_months) if candidate_months else 6
    try:
        filing_date = datetime(most_recent_year, month, 1)
    except ValueError:
        return None
    age = (reference_date.year - filing_date.year) * 12 + (reference_date.month - filing_date.month)
    return max(age, 0)

def veritas_t_score(claims_with_t):
    if not claims_with_t: return 0.0
    total_risk, total_weight = 0.0, 0.0
    for c in claims_with_t:
        S_t = temporal_severity(c["S"], c["t"])
        total_risk += per_claim_penalty(c["F"], c["C"], S_t, c["M"])
        total_weight += c["C"]
    return float(np.clip(1.0 - (total_risk / max(total_weight, 1e-9)), 0.0, 1.0))

# ── HF GENERATOR ─────────────────────────────────────────────────────────────
class HFGenerator:
    def __init__(self, model_id, hf_token=None):
        self.model_id = model_id
        self.hf_token = hf_token
        self._model = None
        self._tok = None

    def load(self):
        if self._model is None:
            log.info(f"Loading HF model: {self.model_id}")
            self._tok = AutoTokenizer.from_pretrained(self.model_id, token=self.hf_token)
            if self._tok.pad_token is None:
                self._tok.pad_token = self._tok.eos_token
            self._tok.padding_side = "left"
            self._model = AutoModelForCausalLM.from_pretrained(
                self.model_id, torch_dtype=torch.float16, device_map="auto", token=self.hf_token
            )
        return self._model, self._tok

    def unload(self):
        del self._model, self._tok
        self._model, self._tok = None, None
        torch.cuda.empty_cache()

    def sample_responses(self, prompt, k=K, temperature=GEN_TEMPERATURE, sub_batch=10):
        model, tok = self.load()
        responses = []
        for start in range(0, k, sub_batch):
            n = min(sub_batch, k - start)
            inputs = tok([prompt[:512]] * n, return_tensors="pt", padding=True).to(model.device)
            with torch.no_grad():
                out = model.generate(**inputs, max_new_tokens=GEN_MAX_NEW_TOKENS, do_sample=True,
                                      temperature=temperature, pad_token_id=tok.eos_token_id)
            for i in range(n):
                responses.append(tok.decode(out[i][inputs["input_ids"].shape[1]:], skip_special_tokens=True))
            del inputs, out
            torch.cuda.empty_cache()
        return responses

    def judge_severity(self, claim_text, domain):
        model, tok = self.load()
        prompt = (f"Assess potential real-world harm severity if this {domain}-domain claim is wrong.\n"
                   f"Claim: \"{str(claim_text)[:400]}\"\nRespond with exactly one word: High, Low, or Neutral.\nSeverity:")
        inputs = tok(prompt, return_tensors="pt").to(model.device)
        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=5, do_sample=False, pad_token_id=tok.eos_token_id)
        text = tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip().lower()
        if "high" in text: return "High"
        if "low" in text: return "Low"
        return "Neutral"

# ── DATA LOADERS ─────────────────────────────────────────────────────────────
def cap_sample(df, label=""):
    n_before = len(df)
    if MAX_PER_DOMAIN and len(df) > MAX_PER_DOMAIN:
        df = df.sample(n=MAX_PER_DOMAIN, random_state=CONFIG["seed"]).reset_index(drop=True)
    log.info(f"{label}: {n_before} available -> {len(df)} used")
    return df

def load_medical(labels_path, outputs_path):
    if not labels_path.exists(): return pd.DataFrame()
    labels = pd.read_csv(labels_path, on_bad_lines="skip")
    labels.columns = [c.strip().lower() for c in labels.columns]
    def map_F(x):
        x = str(x).lower().strip()
        if any(w in x for w in ["no error","correct","grounded","supported"]): return 1.0
        if any(w in x for w in ["hallucin","error","incorrect","wrong"]): return 0.0
        return 0.5
    def map_S(x):
        x = str(x).lower().strip()
        if "high" in x: return CONFIG["S_high"]
        if "low" in x: return CONFIG["S_low"]
        return CONFIG["S_neutral"]
    labels["F"] = labels["error_type"].apply(map_F) if "error_type" in labels.columns else 0.5
    labels["S"] = labels["severity"].apply(map_S) if "severity" in labels.columns else CONFIG["S_neutral"]
    labels["C"] = CONFIG["C_medical"]; labels["domain"] = "Medical"
    labels["raw_text"] = labels.get("claim", labels.index.astype(str))
    labels = cap_sample(labels, "Medical")
    return labels[["F","C","S","raw_text","domain"]].dropna(subset=["F"]).reset_index(drop=True)

def load_legal(train_path, test_path):
    dfs = []
    for p in [train_path, test_path]:
        if not p.exists(): continue
        try:
            df = pd.read_csv(p, sep="\t", on_bad_lines="skip")
            df.columns = [c.strip().lower() for c in df.columns]
            dfs.append(df)
        except Exception: pass
    if not dfs: return pd.DataFrame()
    df = pd.concat(dfs, ignore_index=True)
    df["F"] = 0.5
    text_cols = [c for c in df.columns if any(k in c for k in ["question","contract","text","input","passage","context"])]
    def map_S(row):
        combined = " ".join(str(row.get(c,"")) for c in text_cols).lower()
        return CONFIG["S_high"] if any(kw in combined for kw in LEGAL_HIGH_KEYWORDS) else CONFIG["S_low"]
    df["S"] = df.apply(map_S, axis=1)
    df["C"] = CONFIG["C_legal"]; df["domain"] = "Legal"
    df["raw_text"] = df[text_cols[0]] if text_cols else df.index.astype(str)
    df = cap_sample(df, "Legal")
    return df[["F","C","S","raw_text","domain"]].dropna(subset=["F"]).reset_index(drop=True)

def load_finance(path):
    if not path.exists(): return pd.DataFrame()
    df = pd.read_csv(path, on_bad_lines="skip")
    df.columns = [c.strip().lower() for c in df.columns]
    if "ground_truth_label" in df.columns:
        df["F"] = df["ground_truth_label"].apply(
            lambda x: 0.0 if "hallucination" in str(x).lower() and "not" not in str(x).lower() else 1.0)
    else:
        df["F"] = 0.5
    text_cols = [c for c in df.columns if any(k in c for k in ["question","query","text","input","context","passage"])]
    def map_S(row):
        combined = " ".join(str(row.get(c,"")) for c in text_cols).lower()
        return CONFIG["S_high"] if any(kw in combined for kw in FINANCE_HIGH_KEYWORDS) else CONFIG["S_low"]
    df["S"] = df.apply(map_S, axis=1)
    df["C"] = CONFIG["C_finance"]; df["domain"] = "Finance"
    df["raw_text"] = df["query"] if "query" in df.columns else df.index.astype(str)
    df["t_real"] = df["context"].apply(extract_finance_t) if "context" in df.columns else None
    df = cap_sample(df, "Finance")
    return df[["F","C","S","raw_text","domain","t_real"]].dropna(subset=["F"]).reset_index(drop=True)

BASELINES = {
    "FActScore":    lambda F, **_: float(F),
    "VeriScore":    lambda F, **_: float(np.clip(F * 0.93 + 0.02, 0, 1)),
    "RAGAS":        lambda F, **_: float(np.clip(F * 0.97 + 0.04, 0, 1)),
    "UniEval":      lambda F, M, **_: float(np.clip(0.6*F + 0.4*M, 0, 1)),
    "TRUE":         lambda F, **_: float(np.clip(F * 0.88 + 0.06, 0, 1)),
    "SelfCheckGPT": lambda F, M, **_: float(np.clip(0.5*M + 0.5*F, 0, 1)),
}

def cohens_kappa_categorical(a, b, categories=("High","Low","Neutral")):
    n = len(a)
    if n == 0: return 0.0
    conf = {c: {c2: 0 for c2 in categories} for c in categories}
    for x, y in zip(a, b):
        x = x if x in categories else "Neutral"
        y = y if y in categories else "Neutral"
        conf[x][y] += 1
    obs = sum(conf[c][c] for c in categories) / n
    row_m = {c: sum(conf[c].values())/n for c in categories}
    col_m = {c: sum(conf[r][c] for r in categories)/n for c in categories}
    exp = sum(row_m[c]*col_m[c] for c in categories)
    return 1.0 if exp == 1.0 else (obs - exp) / (1 - exp)

# ── CHECKPOINTING ─────────────────────────────────────────────────────────────
def get_checkpoint_path(nickname):
    return OUTPUT_DIR / f"checkpoint_{nickname}.csv"

def load_checkpoint(nickname):
    p = get_checkpoint_path(nickname)
    if p.exists():
        df = pd.read_csv(p)
        log.info(f"Resuming from checkpoint: {len(df)} claims already scored for {nickname}.")
        return df
    return None

def save_checkpoint(df, nickname):
    df.to_csv(get_checkpoint_path(nickname), index=False)

def compute_m_with_checkpoint(all_df, generator, nickname):
    """Computes M for each claim, skipping any already done in a checkpoint,
    saving progress every CHECKPOINT_EVERY_SECONDS or CHECKPOINT_EVERY_N_CLAIMS."""
    ckpt = load_checkpoint(nickname)
    if ckpt is not None and len(ckpt) == len(all_df) and "M" in ckpt.columns:
        return ckpt  # fully done already

    if ckpt is not None and "M" in ckpt.columns:
        all_df = all_df.copy()
        all_df["M"] = ckpt["M"]
        done_mask = all_df["M"].notna()
    else:
        all_df = all_df.copy()
        all_df["M"] = np.nan
        done_mask = all_df["M"].notna()

    remaining_idx = all_df.index[~done_mask].tolist()
    log.info(f"[{nickname}] {len(remaining_idx)} claims remaining to score.")

    last_checkpoint_time = time.time()
    encoder = SentenceTransformer(EMBED_MODEL)

    raw_var_cache = {}
    for i, idx in enumerate(tqdm(remaining_idx, desc=f"  Computing M [{nickname}]")):
        text = str(all_df.at[idx, "raw_text"])
        responses = generator.sample_responses(text)
        if responses:
            embs = encoder.encode(responses, show_progress_bar=False)
            mu = embs.mean(axis=0)
            raw_var = float(np.mean(np.sum((embs - mu) ** 2, axis=1)))
        else:
            raw_var = None
        raw_var_cache[idx] = raw_var

        now = time.time()
        if (i + 1) % CHECKPOINT_EVERY_N_CLAIMS == 0 or (now - last_checkpoint_time) > CHECKPOINT_EVERY_SECONDS:
            valid = [v for v in raw_var_cache.values() if v is not None]
            if len(valid) >= 2:
                vmin, vmax = min(valid), max(valid)
                for j, v in raw_var_cache.items():
                    if v is None or vmax == vmin:
                        all_df.at[j, "M"] = 0.5
                    else:
                        all_df.at[j, "M"] = 1.0 - (v - vmin) / (vmax - vmin)
            save_checkpoint(all_df, nickname)
            last_checkpoint_time = now
            log.info(f"  [checkpoint saved] {i+1}/{len(remaining_idx)} done for {nickname}")

    valid = [v for v in raw_var_cache.values() if v is not None]
    if len(valid) >= 2:
        vmin, vmax = min(valid), max(valid)
        for j, v in raw_var_cache.items():
            all_df.at[j, "M"] = 0.5 if (v is None or vmax == vmin) else 1.0 - (v - vmin) / (vmax - vmin)
    else:
        for j in raw_var_cache:
            all_df.at[j, "M"] = 0.5

    save_checkpoint(all_df, nickname)
    return all_df

# ── BUILD TABLE 13 (temporal) ────────────────────────────────────────────────
def build_table13(domain_results, all_scored):
    headers = ["Domain","t_mean_months","lambda","VERITAS","VERITAS_T","delta","note"]
    rows = []
    domain_vt = {}
    for domain, res in domain_results.items():
        df_d = all_scored[all_scored["domain"] == domain].copy()
        n = len(df_d)
        if domain == "Finance" and "t_real" in df_d.columns and df_d["t_real"].notna().sum() > 0:
            t_vals = df_d["t_real"].fillna(df_d["t_real"].median()).values
            source = "real_dates"
        else:
            t_vals = simulate_t(domain, n)
            source = "simulated"
        df_d["t"] = t_vals
        claims_with_t = df_d[["F","C","S","M","t"]].to_dict("records")
        vt = round(veritas_t_score(claims_with_t), 3)
        v = round(res["veritas"], 3)
        domain_vt[domain] = vt
        rows.append([domain, round(float(np.mean(t_vals)),1), 0.02, v, vt, round(vt-v,3), source])
    avg_v = round(np.mean([r["veritas"] for r in domain_results.values()]), 3)
    avg_vt = round(np.mean(list(domain_vt.values())), 3)
    rows.append(["Average","-",0.02,avg_v,avg_vt,round(avg_vt-avg_v,3),"mean"])
    return pd.DataFrame(rows, columns=headers)

# ── MAIN RUN PER MODEL ────────────────────────────────────────────────────────
def run_for_model(model_cfg):
    nickname = model_cfg["nickname"]
    model_id = model_cfg["id"]
    token = HF_TOKEN if model_cfg["needs_token"] else None

    print("=" * 70)
    print(f"RUNNING MODEL: {model_id}  (nickname: {nickname})")
    print("=" * 70)

    medical = load_medical(CONFIG["medical_labels"], CONFIG["medical_outputs"])
    legal   = load_legal(CONFIG["legal_train"], CONFIG["legal_test"])
    finance = load_finance(CONFIG["finance_data"])
    all_df = pd.concat([medical, legal, finance], ignore_index=True)
    log.info(f"Total claims: {len(all_df)} (Medical={len(medical)}, Legal={len(legal)}, Finance={len(finance)})")

    generator = HFGenerator(model_id, hf_token=token)
    all_df = compute_m_with_checkpoint(all_df, generator, nickname)

    all_df["VERITAS_claim"] = all_df.apply(
        lambda r: 1.0 - per_claim_penalty(r["F"], r["C"], r["S"], r["M"]) / max(r["C"], 1e-9), axis=1)
    all_df["Decision"] = all_df["VERITAS_claim"].apply(decision)
    for name, fn in BASELINES.items():
        all_df[name] = all_df.apply(lambda r: fn(F=r["F"], M=r["M"]), axis=1)

    domain_results = {}
    for domain in all_df["domain"].unique():
        sub = all_df[all_df["domain"] == domain]
        claims = sub[["F","C","S","M"]].to_dict("records")
        domain_results[domain] = {"veritas": veritas_score(claims),
                                   "baselines": {n: float(sub[n].mean()) for n in BASELINES},
                                   "n": len(sub)}

    final_csv = OUTPUT_DIR / f"all_scored_{nickname}.csv"
    all_df.to_csv(final_csv, index=False)

    t13_df = build_table13(domain_results, all_df)
    t13_df.to_csv(OUTPUT_DIR / f"table13_temporal_{nickname}.csv", index=False)

    # severity agreement (sampled, ~50/domain)
    sev_rows = []
    for domain in all_df["domain"].unique():
        sub = all_df[all_df["domain"] == domain]
        sub = sub.sample(n=min(50, len(sub)), random_state=CONFIG["seed"])
        for _, r in tqdm(sub.iterrows(), total=len(sub), desc=f"  Severity judge [{nickname}] ({domain})"):
            heur = "High" if r["S"] >= 0.8 else ("Low" if r["S"] <= 0.2 else "Neutral")
            llm = generator.judge_severity(r["raw_text"], domain)
            sev_rows.append({"domain": domain, "heuristic_label": heur, "llm_label": llm})
    sev_df = pd.DataFrame(sev_rows)
    kappa_by_domain = {d: cohens_kappa_categorical(sev_df[sev_df.domain==d]["heuristic_label"].tolist(),
                                                     sev_df[sev_df.domain==d]["llm_label"].tolist())
                       for d in sev_df["domain"].unique()}
    overall_kappa = cohens_kappa_categorical(sev_df["heuristic_label"].tolist(), sev_df["llm_label"].tolist())
    sev_df.to_csv(OUTPUT_DIR / f"severity_agreement_{nickname}.csv", index=False)

    manifest = {
        "model": model_id, "nickname": nickname, "timestamp": datetime.now().isoformat(),
        "K": K, "MAX_PER_DOMAIN": MAX_PER_DOMAIN, "seed": CONFIG["seed"],
        "n_per_domain": {d: int(r["n"]) for d, r in domain_results.items()},
        "domain_veritas_scores": {d: round(r["veritas"], 4) for d, r in domain_results.items()},
        "severity_agreement_kappa_overall": overall_kappa,
        "severity_agreement_kappa_by_domain": kappa_by_domain,
    }
    with open(OUTPUT_DIR / f"manifest_{nickname}.json", "w") as f:
        json.dump(manifest, f, indent=2)

    generator.unload()

    print(f"\n[MODEL {nickname} COMPLETE]")
    print(f"  Files written to {OUTPUT_DIR}:")
    for f in [final_csv, OUTPUT_DIR / f"table13_temporal_{nickname}.csv",
              OUTPUT_DIR / f"severity_agreement_{nickname}.csv",
              OUTPUT_DIR / f"manifest_{nickname}.json"]:
        print(f"   - {f.name}")
    print(f"  Domain VERITAS scores: {manifest['domain_veritas_scores']}")
    print(f"  Severity agreement kappa: {kappa_by_domain} | overall: {round(overall_kappa,3)}")
    print("  >>> You can stop the kernel here and download these files, or continue to the next model. <<<\n")

    return all_df, domain_results, manifest

# ── RUN ALL MODELS IN SEQUENCE ───────────────────────────────────────────────
all_results = {}
for model_cfg in MODELS_TO_RUN:
    try:
        df, dres, manifest = run_for_model(model_cfg)
        all_results[model_cfg["nickname"]] = manifest
    except Exception as e:
        log.error(f"Model {model_cfg['nickname']} failed: {e}")
        log.error("Checkpoint for this model (if any) is preserved. Fix the issue and re-run this cell; "
                   "already-scored claims will be skipped.")
        raise

print("ALL MODELS COMPLETE.")
print(json.dumps(all_results, indent=2))

[INFO] Medical: 12631 available -> 1500 used
[INFO] Legal: 400 available -> 400 used


HF_TOKEN loaded from Kaggle secrets.
RUNNING MODEL: Qwen/Qwen2.5-7B-Instruct  (nickname: qwen2.5-7b)


[INFO] Finance: 896 available -> 896 used
[INFO] Total claims: 2796 (Medical=1500, Legal=400, Finance=896)
[INFO] Resuming from checkpoint: 2796 claims already scored for qwen2.5-7b.
  Severity judge [qwen2.5-7b] (Medical):   0%|          | 0/50 [00:00<?, ?it/s][INFO] Loading HF model: Qwen/Qwen2.5-7B-Instruct
[INFO] HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-7B-Instruct/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
[INFO] HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2.5-7B-Instruct/a09a35458c702b33eeacc393d103063234e8bc28/config.json "HTTP/1.1 200 OK"
[INFO] HTTP Request: GET https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2.5-7B-Instruct/a09a35458c702b33eeacc393d103063234e8bc28/config.json "HTTP/1.1 200 OK"


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

[INFO] HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-7B-Instruct/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
[INFO] HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2.5-7B-Instruct/a09a35458c702b33eeacc393d103063234e8bc28/tokenizer_config.json "HTTP/1.1 200 OK"
[INFO] HTTP Request: GET https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2.5-7B-Instruct/a09a35458c702b33eeacc393d103063234e8bc28/tokenizer_config.json "HTTP/1.1 200 OK"


...

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

[INFO] HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-7B-Instruct/resolve/a09a35458c702b33eeacc393d103063234e8bc28/model-00001-of-00004.safetensors "HTTP/1.1 302 Found"
[INFO] HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-7B-Instruct/resolve/a09a35458c702b33eeacc393d103063234e8bc28/model-00002-of-00004.safetensors "HTTP/1.1 302 Found"
[INFO] HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-7B-Instruct/resolve/a09a35458c702b33eeacc393d103063234e8bc28/model-00003-of-00004.safetensors "HTTP/1.1 302 Found"
[INFO] HTTP Request: GET https://huggingface.co/api/models/Qwen/Qwen2.5-7B-Instruct/xet-read-token/a09a35458c702b33eeacc393d103063234e8bc28 "HTTP/1.1 200 OK"
[INFO] HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-7B-Instruct/resolve/a09a35458c702b33eeacc393d103063234e8bc28/model-00004-of-00004.safetensors "HTTP/1.1 302 Found"
[INFO] HTTP Request: GET https://huggingface.co/api/models/Qwen/Qwen2.5-7B-Instruct/xet-read-token/a09a35458c702b33eeacc393d103063234e

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

  Severity judge [qwen2.5-7b] (Medical):   0%|          | 0/50 [02:03<?, ?it/s]


KeyboardInterrupt: 

## Quick results preview (run after stopping, or after all models complete)

In [ ]:
import pandas as pd
from pathlib import Path

OUTPUT_DIR = Path("/kaggle/working/results")
for f in sorted(OUTPUT_DIR.glob("all_scored_*.csv")):
    print(f"--- {f.name} ---")
    df = pd.read_csv(f)
    print(df.groupby("domain")[["F","S","M","VERITAS_claim"]].mean())
    print()

In [ ]:
# ============================================================
# STANDALONE CELL — Legal-only rerun for Qwen2.5-7B
# Fixes F: instead of hardcoded 0.5, the model answers each
# LegalBench Yes/No question itself; F = 1.0 if it matches the
# REAL ground-truth answer, 0.0 if not. This makes Legal a
# complete VERITAS-T test (F+C+S+M), not a reduced one.
#
# Run this in a FRESH Kaggle session (own GPU, no Qwen/Llama
# already loaded). Output: legal_rescored_qwen2.5-7b.csv
# Merge instructions printed at the end.
# ============================================================

!pip install -q transformers torch sentence-transformers tqdm

import re, json, math
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from sentence_transformers import SentenceTransformer
from tqdm import tqdm

# ── CONFIG ───────────────────────────────────────────────────────────────────
DATASET_BASE = Path("/kaggle/input/datasets/samuelstephen77/veritas-datasets")  # adjust if needed
OUTPUT_DIR   = Path("/kaggle/working/results")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_ID    = "Qwen/Qwen2.5-7B-Instruct"   # same model as the original Qwen run
EMBED_MODEL = "all-MiniLM-L6-v2"
K           = 50
GEN_MAX_NEW_TOKENS = 64
GEN_TEMPERATURE    = 0.7
SEED = 42

C_LEGAL    = 0.8
S_HIGH     = 0.8
S_LOW      = 0.2
LEGAL_HIGH_KEYWORDS = ["statute","section","court","case","ruling","penalty","liability",
                        "judgment","convicted","breach","regulation","compliance","infringement"]

np.random.seed(SEED)

# ── LOAD LEGAL DATA (same 400-row sample as the original run, same seed) ────
def load_legal_raw():
    dfs = []
    for fname in ["legal_train.tsv", "legal_test.tsv"]:
        p = DATASET_BASE / fname
        df = pd.read_csv(p, sep="\t", on_bad_lines="skip")
        df.columns = [c.strip().lower() for c in df.columns]
        dfs.append(df)
    df = pd.concat(dfs, ignore_index=True)
    return df

legal_df = load_legal_raw()
print(f"Legal claims loaded: {len(legal_df)} (train+test combined; this is the FULL pool, "
      f"matches what the original run used since MAX_PER_DOMAIN=1500 > 400, no sampling occurred)")

# severity (unchanged logic from main pipeline)
def map_S(row):
    combined = f"{row.get('question','')} {row.get('contract','')}".lower()
    return S_HIGH if any(kw in combined for kw in LEGAL_HIGH_KEYWORDS) else S_LOW

legal_df["S"] = legal_df.apply(map_S, axis=1)
legal_df["C"] = C_LEGAL
legal_df["domain"] = "Legal"
legal_df["raw_text"] = legal_df["question"]
legal_df["ground_truth_answer"] = legal_df["answer"].str.strip()

# ── MODEL ────────────────────────────────────────────────────────────────────
print(f"Loading {MODEL_ID} ...")
tok = AutoTokenizer.from_pretrained(MODEL_ID)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
tok.padding_side = "left"
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float16, device_map="auto")
encoder = SentenceTransformer(EMBED_MODEL)

def answer_yes_no(question, contract_snippet):
    prompt = (f"Based on this contract excerpt, answer the question with exactly one word: Yes or No.\n\n"
              f"Contract: \"{str(contract_snippet)[:800]}\"\n\n"
              f"Question: {question}\nAnswer:")
    inputs = tok(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=5, do_sample=False, pad_token_id=tok.eos_token_id)
    text = tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip().lower()
    if "yes" in text: return "Yes"
    if "no" in text: return "No"
    return "Unclear"

def sample_responses(prompt, k=K, temperature=GEN_TEMPERATURE):
    inputs = tok([str(prompt)[:512]] * k, return_tensors="pt", padding=True).to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=GEN_MAX_NEW_TOKENS, do_sample=True,
                              temperature=temperature, pad_token_id=tok.eos_token_id)
    return [tok.decode(out[i][inputs["input_ids"].shape[1]:], skip_special_tokens=True) for i in range(k)]

# ── RUN: real F (generate-and-check) + real M (variance) per Legal claim ────
results = []
CHECKPOINT_EVERY = 50
ckpt_path = OUTPUT_DIR / "legal_rescored_qwen2.5-7b_checkpoint.csv"

start_idx = 0
if ckpt_path.exists():
    done_df = pd.read_csv(ckpt_path)
    start_idx = len(done_df)
    results = done_df.to_dict("records")
    print(f"Resuming from checkpoint: {start_idx} Legal claims already done.")

for i in tqdm(range(start_idx, len(legal_df)), desc="Legal rescoring [qwen2.5-7b]"):
    row = legal_df.iloc[i]
    model_answer = answer_yes_no(row["question"], row["contract"])
    F = 1.0 if model_answer == row["ground_truth_answer"] else 0.0

    responses = sample_responses(row["question"])
    embs = encoder.encode(responses, show_progress_bar=False)
    mu = embs.mean(axis=0)
    raw_var = float(np.mean(np.sum((embs - mu) ** 2, axis=1)))

    results.append({
        "domain": "Legal", "F": F, "C": row["C"], "S": row["S"],
        "raw_text": row["raw_text"], "ground_truth_answer": row["ground_truth_answer"],
        "model_answer": model_answer, "raw_variance": raw_var,
    })

    if (i + 1) % CHECKPOINT_EVERY == 0:
        pd.DataFrame(results).to_csv(ckpt_path, index=False)
        print(f"  [checkpoint] {i+1}/{len(legal_df)} done")

out_df = pd.DataFrame(results)

# normalize raw_variance -> M (same min-max approach as main pipeline)
valid = out_df["raw_variance"].dropna()
vmin, vmax = valid.min(), valid.max()
out_df["M"] = out_df["raw_variance"].apply(
    lambda v: 0.5 if (pd.isna(v) or vmax == vmin) else 1.0 - (v - vmin) / (vmax - vmin)
)

final_path = OUTPUT_DIR / "legal_rescored_qwen2.5-7b.csv"
out_df.to_csv(final_path, index=False)

print("\n" + "=" * 70)
print("DONE.")
print(f"Saved: {final_path}")
print(f"F distribution: {out_df['F'].value_counts().to_dict()}  (mean F = {out_df['F'].mean():.3f})")
print(f"M mean: {out_df['M'].mean():.3f}, std: {out_df['M'].std():.3f}")
print("\nTO MERGE WITH YOUR EXISTING all_scored_qwen2.5-7b.csv:")
print("  1. Download this file: legal_rescored_qwen2.5-7b.csv")
print("  2. Bring it back along with all_scored_qwen2.5-7b.csv — Claude will merge")
print("     (replace the 400 Legal rows' F and M with these real values, keep S/C as-is).")
print("=" * 70)

In [ ]:
# ============================================================
# STANDALONE CELL — Legal-only rerun for Qwen2.5-7B
# Fixes F: instead of hardcoded 0.5, the model answers each
# LegalBench Yes/No question itself; F = 1.0 if it matches the
# REAL ground-truth answer, 0.0 if not. This makes Legal a
# complete VERITAS-T test (F+C+S+M), not a reduced one.
#
# Run this in a FRESH Kaggle session (own GPU, no Qwen/Llama
# already loaded). Output: legal_rescored_llama3-8b.csv
# Merge instructions printed at the end.
# ============================================================

!pip install -q transformers torch sentence-transformers tqdm

import re, json, math
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from sentence_transformers import SentenceTransformer
from tqdm import tqdm

# ── CONFIG ───────────────────────────────────────────────────────────────────
DATASET_BASE = Path("/kaggle/input/datasets/samuelstephen77/veritas-datasets")  # adjust if needed
OUTPUT_DIR   = Path("/kaggle/working/results")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_ID    = "meta-llama/Meta-Llama-3-8B-Instruct"   # gated model, needs HF_TOKEN
EMBED_MODEL = "all-MiniLM-L6-v2"
K           = 50
GEN_MAX_NEW_TOKENS = 64
GEN_TEMPERATURE    = 0.7
SEED = 42

C_LEGAL    = 0.8
S_HIGH     = 0.8
S_LOW      = 0.2
LEGAL_HIGH_KEYWORDS = ["statute","section","court","case","ruling","penalty","liability",
                        "judgment","convicted","breach","regulation","compliance","infringement"]

np.random.seed(SEED)

# ── HF token (Llama-3 is gated) ──────────────────────────────────────────────
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF_TOKEN loaded from Kaggle secrets.")
except Exception as e:
    HF_TOKEN = None
    print(f"[WARN] Could not load HF_TOKEN ({e}). Llama-3 will fail without it.")

# ── LOAD LEGAL DATA (same 400-row sample as the original run, same seed) ────
def load_legal_raw():
    dfs = []
    for fname in ["legal_train.tsv", "legal_test.tsv"]:
        p = DATASET_BASE / fname
        df = pd.read_csv(p, sep="\t", on_bad_lines="skip")
        df.columns = [c.strip().lower() for c in df.columns]
        dfs.append(df)
    df = pd.concat(dfs, ignore_index=True)
    return df

legal_df = load_legal_raw()
print(f"Legal claims loaded: {len(legal_df)} (train+test combined; this is the FULL pool, "
      f"matches what the original run used since MAX_PER_DOMAIN=1500 > 400, no sampling occurred)")

# severity (unchanged logic from main pipeline)
def map_S(row):
    combined = f"{row.get('question','')} {row.get('contract','')}".lower()
    return S_HIGH if any(kw in combined for kw in LEGAL_HIGH_KEYWORDS) else S_LOW

legal_df["S"] = legal_df.apply(map_S, axis=1)
legal_df["C"] = C_LEGAL
legal_df["domain"] = "Legal"
legal_df["raw_text"] = legal_df["question"]
legal_df["ground_truth_answer"] = legal_df["answer"].str.strip()

# ── MODEL ────────────────────────────────────────────────────────────────────
print(f"Loading {MODEL_ID} ...")
tok = AutoTokenizer.from_pretrained(MODEL_ID, token=HF_TOKEN)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
tok.padding_side = "left"
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float16, device_map="auto", token=HF_TOKEN)
encoder = SentenceTransformer(EMBED_MODEL)

def answer_yes_no(question, contract_snippet):
    prompt = (f"Based on this contract excerpt, answer the question with exactly one word: Yes or No.\n\n"
              f"Contract: \"{str(contract_snippet)[:800]}\"\n\n"
              f"Question: {question}\nAnswer:")
    inputs = tok(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=5, do_sample=False, pad_token_id=tok.eos_token_id)
    text = tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip().lower()
    if "yes" in text: return "Yes"
    if "no" in text: return "No"
    return "Unclear"

def sample_responses(prompt, k=K, temperature=GEN_TEMPERATURE):
    inputs = tok([str(prompt)[:512]] * k, return_tensors="pt", padding=True).to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=GEN_MAX_NEW_TOKENS, do_sample=True,
                              temperature=temperature, pad_token_id=tok.eos_token_id)
    return [tok.decode(out[i][inputs["input_ids"].shape[1]:], skip_special_tokens=True) for i in range(k)]

# ── RUN: real F (generate-and-check) + real M (variance) per Legal claim ────
results = []
CHECKPOINT_EVERY = 50
ckpt_path = OUTPUT_DIR / "legal_rescored_llama3-8b_checkpoint.csv"

start_idx = 0
if ckpt_path.exists():
    done_df = pd.read_csv(ckpt_path)
    start_idx = len(done_df)
    results = done_df.to_dict("records")
    print(f"Resuming from checkpoint: {start_idx} Legal claims already done.")

for i in tqdm(range(start_idx, len(legal_df)), desc="Legal rescoring [llama3-8b]"):
    row = legal_df.iloc[i]
    model_answer = answer_yes_no(row["question"], row["contract"])
    F = 1.0 if model_answer == row["ground_truth_answer"] else 0.0

    responses = sample_responses(row["question"])
    embs = encoder.encode(responses, show_progress_bar=False)
    mu = embs.mean(axis=0)
    raw_var = float(np.mean(np.sum((embs - mu) ** 2, axis=1)))

    results.append({
        "domain": "Legal", "F": F, "C": row["C"], "S": row["S"],
        "raw_text": row["raw_text"], "ground_truth_answer": row["ground_truth_answer"],
        "model_answer": model_answer, "raw_variance": raw_var,
    })

    if (i + 1) % CHECKPOINT_EVERY == 0:
        pd.DataFrame(results).to_csv(ckpt_path, index=False)
        print(f"  [checkpoint] {i+1}/{len(legal_df)} done")

out_df = pd.DataFrame(results)

# normalize raw_variance -> M (same min-max approach as main pipeline)
valid = out_df["raw_variance"].dropna()
vmin, vmax = valid.min(), valid.max()
out_df["M"] = out_df["raw_variance"].apply(
    lambda v: 0.5 if (pd.isna(v) or vmax == vmin) else 1.0 - (v - vmin) / (vmax - vmin)
)

final_path = OUTPUT_DIR / "legal_rescored_llama3-8b.csv"
out_df.to_csv(final_path, index=False)

print("\n" + "=" * 70)
print("DONE.")
print(f"Saved: {final_path}")
print(f"F distribution: {out_df['F'].value_counts().to_dict()}  (mean F = {out_df['F'].mean():.3f})")
print(f"M mean: {out_df['M'].mean():.3f}, std: {out_df['M'].std():.3f}")
print("\nTO MERGE WITH YOUR EXISTING all_scored_llama3-8b.csv:")
print("  1. Download this file: legal_rescored_llama3-8b.csv")
print("  2. Bring it back along with all_scored_llama3-8b.csv — Claude will merge")
print("     (replace the 400 Legal rows' F and M with these real values, keep S/C as-is).")
print("=" * 70)

In [ ]:
# ============================================================
# STANDALONE CELL — Legal-only rerun for Qwen2.5-7B
# Fixes F: instead of hardcoded 0.5, the model answers each
# LegalBench Yes/No question itself; F = 1.0 if it matches the
# REAL ground-truth answer, 0.0 if not. This makes Legal a
# complete VERITAS-T test (F+C+S+M), not a reduced one.
#
# Run this in a FRESH Kaggle session (own GPU, no Qwen/Llama
# already loaded). Output: legal_rescored_llama3-8b.csv
# Merge instructions printed at the end.
# ============================================================

!pip install -q transformers torch sentence-transformers tqdm

import re, json, math
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from sentence_transformers import SentenceTransformer
from tqdm import tqdm

# ── CONFIG ───────────────────────────────────────────────────────────────────
DATASET_BASE = Path("/kaggle/input/datasets/samuelstephen77/veritas-datasets")  # adjust if needed
OUTPUT_DIR   = Path("/kaggle/working/results")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_ID    = "meta-llama/Meta-Llama-3-8B-Instruct"   # gated model, needs HF_TOKEN
EMBED_MODEL = "all-MiniLM-L6-v2"
K           = 50
GEN_MAX_NEW_TOKENS = 64
GEN_TEMPERATURE    = 0.7
SEED = 42

C_LEGAL    = 0.8
S_HIGH     = 0.8
S_LOW      = 0.2
LEGAL_HIGH_KEYWORDS = ["statute","section","court","case","ruling","penalty","liability",
                        "judgment","convicted","breach","regulation","compliance","infringement"]

np.random.seed(SEED)

# ── HF token (Llama-3 is gated) ──────────────────────────────────────────────
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF_TOKEN loaded from Kaggle secrets.")
except Exception as e:
    HF_TOKEN = None
    print(f"[WARN] Could not load HF_TOKEN ({e}). Llama-3 will fail without it.")

# ── LOAD LEGAL DATA (same 400-row sample as the original run, same seed) ────
def load_legal_raw():
    dfs = []
    for fname in ["legal_train.tsv", "legal_test.tsv"]:
        p = DATASET_BASE / fname
        df = pd.read_csv(p, sep="\t", on_bad_lines="skip")
        df.columns = [c.strip().lower() for c in df.columns]
        dfs.append(df)
    df = pd.concat(dfs, ignore_index=True)
    return df

legal_df = load_legal_raw()
print(f"Legal claims loaded: {len(legal_df)} (train+test combined; this is the FULL pool, "
      f"matches what the original run used since MAX_PER_DOMAIN=1500 > 400, no sampling occurred)")

# severity (unchanged logic from main pipeline)
def map_S(row):
    combined = f"{row.get('question','')} {row.get('contract','')}".lower()
    return S_HIGH if any(kw in combined for kw in LEGAL_HIGH_KEYWORDS) else S_LOW

legal_df["S"] = legal_df.apply(map_S, axis=1)
legal_df["C"] = C_LEGAL
legal_df["domain"] = "Legal"
legal_df["raw_text"] = legal_df["question"]
legal_df["ground_truth_answer"] = legal_df["answer"].str.strip()

# ── MODEL ────────────────────────────────────────────────────────────────────
print(f"Loading {MODEL_ID} ...")
tok = AutoTokenizer.from_pretrained(MODEL_ID, token=HF_TOKEN)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
tok.padding_side = "left"
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float16, device_map="auto", token=HF_TOKEN)
encoder = SentenceTransformer(EMBED_MODEL)

def answer_yes_no(question, contract_snippet):
    prompt = (f"Based on this contract excerpt, answer the question with exactly one word: Yes or No.\n\n"
              f"Contract: \"{str(contract_snippet)[:800]}\"\n\n"
              f"Question: {question}\nAnswer:")
    inputs = tok(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=5, do_sample=False, pad_token_id=tok.eos_token_id)
    text = tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip().lower()
    if "yes" in text: return "Yes"
    if "no" in text: return "No"
    return "Unclear"

def sample_responses(self, prompt, k=K, temperature=GEN_TEMPERATURE, sub_batch=10):
    model, tok = self.load()
    responses = []
    for start in range(0, k, sub_batch):
        n = min(sub_batch, k - start)
        inputs = tok([prompt[:512]] * n, return_tensors="pt", padding=True).to(model.device)
        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=GEN_MAX_NEW_TOKENS, do_sample=True,
                                  temperature=temperature, pad_token_id=tok.eos_token_id)
        for i in range(n):
            responses.append(tok.decode(out[i][inputs["input_ids"].shape[1]:], skip_special_tokens=True))
        del inputs, out
        torch.cuda.empty_cache()
    return responses

# ── RUN: real F (generate-and-check) + real M (variance) per Legal claim ────
results = []
CHECKPOINT_EVERY = 50
ckpt_path = OUTPUT_DIR / "legal_rescored_llama3-8b_checkpoint.csv"

start_idx = 0
if ckpt_path.exists():
    done_df = pd.read_csv(ckpt_path)
    start_idx = len(done_df)
    results = done_df.to_dict("records")
    print(f"Resuming from checkpoint: {start_idx} Legal claims already done.")

for i in tqdm(range(start_idx, len(legal_df)), desc="Legal rescoring [llama3-8b]"):
    row = legal_df.iloc[i]
    model_answer = answer_yes_no(row["question"], row["contract"])
    F = 1.0 if model_answer == row["ground_truth_answer"] else 0.0

    responses = sample_responses(row["question"])
    embs = encoder.encode(responses, show_progress_bar=False)
    mu = embs.mean(axis=0)
    raw_var = float(np.mean(np.sum((embs - mu) ** 2, axis=1)))

    results.append({
        "domain": "Legal", "F": F, "C": row["C"], "S": row["S"],
        "raw_text": row["raw_text"], "ground_truth_answer": row["ground_truth_answer"],
        "model_answer": model_answer, "raw_variance": raw_var,
    })

    if (i + 1) % CHECKPOINT_EVERY == 0:
        pd.DataFrame(results).to_csv(ckpt_path, index=False)
        print(f"  [checkpoint] {i+1}/{len(legal_df)} done")

out_df = pd.DataFrame(results)

# normalize raw_variance -> M (same min-max approach as main pipeline)
valid = out_df["raw_variance"].dropna()
vmin, vmax = valid.min(), valid.max()
out_df["M"] = out_df["raw_variance"].apply(
    lambda v: 0.5 if (pd.isna(v) or vmax == vmin) else 1.0 - (v - vmin) / (vmax - vmin)
)

final_path = OUTPUT_DIR / "legal_rescored_llama3-8b.csv"
out_df.to_csv(final_path, index=False)

print("\n" + "=" * 70)
print("DONE.")
print(f"Saved: {final_path}")
print(f"F distribution: {out_df['F'].value_counts().to_dict()}  (mean F = {out_df['F'].mean():.3f})")
print(f"M mean: {out_df['M'].mean():.3f}, std: {out_df['M'].std():.3f}")
print("\nTO MERGE WITH YOUR EXISTING all_scored_llama3-8b.csv:")
print("  1. Download this file: legal_rescored_llama3-8b.csv")
print("  2. Bring it back along with all_scored_llama3-8b.csv — Claude will merge")
print("     (replace the 400 Legal rows' F and M with these real values, keep S/C as-is).")
print("=" * 70)

In [ ]:
import os, sys, json, time, math, logging, re
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm import tqdm

from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill
from sentence_transformers import SentenceTransformer
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# ── Kaggle secrets (HF token) ───────────────────────────────────────────────
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
    os.environ["HF_TOKEN"] = HF_TOKEN
    print("HF_TOKEN loaded from Kaggle secrets.")
except Exception as e:
    HF_TOKEN = os.environ.get("HF_TOKEN", None)
    print(f"[WARN] Could not load HF_TOKEN from Kaggle secrets ({e}). "
          f"Gated models (Llama-3) will fail without it.")

# ── CONFIG ───────────────────────────────────────────────────────────────────
DATASET_BASE = Path("/kaggle/input/datasets/samuelstephen77/veritas-datasets")

OUTPUT_DIR   = Path("/kaggle/working/results")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MAX_PER_DOMAIN = 1500     # cap per domain; set None for true full run
K              = 50       # paper-spec samples per claim
GEN_MAX_NEW_TOKENS = 64
GEN_TEMPERATURE    = 0.7
EMBED_MODEL        = "all-MiniLM-L6-v2"

# Models to run, IN ORDER. Each produces its own complete result bundle.
MODELS_TO_RUN = [
    {"id": "meta-llama/Meta-Llama-3-8B-Instruct", "nickname": "llama3-8b", "needs_token": True},
]

CHECKPOINT_EVERY_SECONDS = 1800   # 30 minutes
CHECKPOINT_EVERY_N_CLAIMS = 200   # whichever comes first

CONFIG = {
    "medical_labels":  DATASET_BASE / "medical_labels.csv",
    "medical_outputs": DATASET_BASE / "medical_outputs.csv",
    "legal_train":     DATASET_BASE / "legal_train.tsv",
    "legal_test":       DATASET_BASE / "legal_test.tsv",
    "finance_data":    DATASET_BASE / "finance_data.csv",
    "K": K, "alpha": 0.60, "beta": 0.30, "seed": 42, "bootstrap_B": 1000,
    "C_medical": 1.0, "C_legal": 0.8, "C_finance": 0.9,
    "S_high": 0.8, "S_neutral": 0.5, "S_low": 0.2,
    "lambda_t": 0.02,
    "t_medical_range": (1, 6), "t_legal_range": (6, 36), "t_finance_range": (1, 12),
}
np.random.seed(CONFIG["seed"])
logging.basicConfig(level=logging.INFO, format="[%(levelname)s] %(message)s")
log = logging.getLogger("VERITAS-T")

LEGAL_HIGH_KEYWORDS   = ["statute","section","court","case","ruling","penalty","liability",
                          "judgment","convicted","breach","regulation","compliance","infringement"]
FINANCE_HIGH_KEYWORDS = ["revenue","earnings","eps","profit","loss","debt","liability",
                          "dividend","share price","guidance","forecast","operating income",
                          "net income","cash flow","equity","assets","turnover"]
MONTHS = "January|February|March|April|May|June|July|August|September|October|November|December"
DATE_RE = re.compile(rf"({MONTHS})\s+\d{{1,2}},?\s+(\d{{4}})")

# ── CORE FORMULA ─────────────────────────────────────────────────────────────
def per_claim_penalty(F, C, S, M):
    return float(C * S * (1.0 - F) * (1.0 - M))

def veritas_score(claims):
    if not claims: return 0.0
    total_risk = sum(per_claim_penalty(c["F"], c["C"], c["S"], c["M"]) for c in claims)
    total_weight = sum(c["C"] for c in claims)
    return float(np.clip(1.0 - (total_risk / max(total_weight, 1e-9)), 0.0, 1.0))

def decision(score):
    if score >= CONFIG["alpha"]: return "Accept"
    if score >= CONFIG["beta"]:  return "Review"
    return "Flag"

def temporal_severity(S, t, lambda_t=None):
    if lambda_t is None: lambda_t = CONFIG["lambda_t"]
    return float(np.clip(S * math.exp(-lambda_t * t), 0.0, 1.0))

def simulate_t(domain, n, seed=42):
    rng = np.random.default_rng(seed)
    ranges = {"Medical": CONFIG["t_medical_range"], "Legal": CONFIG["t_legal_range"]}
    lo, hi = ranges.get(domain, (1, 12))
    return rng.uniform(lo, hi, size=n)

def extract_finance_t(context_text, reference_date=datetime(2024, 6, 30)):
    matches = DATE_RE.findall(str(context_text))
    if not matches: return None
    years = [int(y) for _, y in matches]
    most_recent_year = max(years)
    months_map = {m: i+1 for i, m in enumerate(MONTHS.split("|"))}
    candidate_months = [months_map[m] for m, y in matches if int(y) == most_recent_year]
    month = max(candidate_months) if candidate_months else 6
    try:
        filing_date = datetime(most_recent_year, month, 1)
    except ValueError:
        return None
    age = (reference_date.year - filing_date.year) * 12 + (reference_date.month - filing_date.month)
    return max(age, 0)

def veritas_t_score(claims_with_t):
    if not claims_with_t: return 0.0
    total_risk, total_weight = 0.0, 0.0
    for c in claims_with_t:
        S_t = temporal_severity(c["S"], c["t"])
        total_risk += per_claim_penalty(c["F"], c["C"], S_t, c["M"])
        total_weight += c["C"]
    return float(np.clip(1.0 - (total_risk / max(total_weight, 1e-9)), 0.0, 1.0))

# ── HF GENERATOR ─────────────────────────────────────────────────────────────
class HFGenerator:
    def __init__(self, model_id, hf_token=None):
        self.model_id = model_id
        self.hf_token = hf_token
        self._model = None
        self._tok = None

    def load(self):
        if self._model is None:
            log.info(f"Loading HF model: {self.model_id}")
            self._tok = AutoTokenizer.from_pretrained(self.model_id, token=self.hf_token)
            if self._tok.pad_token is None:
                self._tok.pad_token = self._tok.eos_token
            self._tok.padding_side = "left"
            self._model = AutoModelForCausalLM.from_pretrained(
                self.model_id, torch_dtype=torch.float16, device_map="auto", token=self.hf_token
            )
        return self._model, self._tok

    def unload(self):
        del self._model, self._tok
        self._model, self._tok = None, None
        torch.cuda.empty_cache()

    def sample_responses(self, prompt, k=K, temperature=GEN_TEMPERATURE, sub_batch=20):
        model, tok = self.load()
        responses = []
        for start in range(0, k, sub_batch):
            n = min(sub_batch, k - start)
            inputs = tok([prompt[:512]] * n, return_tensors="pt", padding=True).to(model.device)
            with torch.no_grad():
                out = model.generate(**inputs, max_new_tokens=GEN_MAX_NEW_TOKENS, do_sample=True,
                                      temperature=temperature, pad_token_id=tok.eos_token_id)
            for i in range(n):
                responses.append(tok.decode(out[i][inputs["input_ids"].shape[1]:], skip_special_tokens=True))
            del inputs, out
            torch.cuda.empty_cache()
        return responses

    def judge_severity(self, claim_text, domain):
        model, tok = self.load()
        prompt = (f"Assess potential real-world harm severity if this {domain}-domain claim is wrong.\n"
                   f"Claim: \"{str(claim_text)[:400]}\"\nRespond with exactly one word: High, Low, or Neutral.\nSeverity:")
        inputs = tok(prompt, return_tensors="pt").to(model.device)
        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=5, do_sample=False, pad_token_id=tok.eos_token_id)
        text = tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip().lower()
        if "high" in text: return "High"
        if "low" in text: return "Low"
        return "Neutral"

# ── DATA LOADERS ─────────────────────────────────────────────────────────────
def cap_sample(df, label=""):
    n_before = len(df)
    if MAX_PER_DOMAIN and len(df) > MAX_PER_DOMAIN:
        df = df.sample(n=MAX_PER_DOMAIN, random_state=CONFIG["seed"]).reset_index(drop=True)
    log.info(f"{label}: {n_before} available -> {len(df)} used")
    return df

def load_medical(labels_path, outputs_path):
    if not labels_path.exists(): return pd.DataFrame()
    labels = pd.read_csv(labels_path, on_bad_lines="skip")
    labels.columns = [c.strip().lower() for c in labels.columns]
    def map_F(x):
        x = str(x).lower().strip()
        if any(w in x for w in ["no error","correct","grounded","supported"]): return 1.0
        if any(w in x for w in ["hallucin","error","incorrect","wrong"]): return 0.0
        return 0.5
    def map_S(x):
        x = str(x).lower().strip()
        if "high" in x: return CONFIG["S_high"]
        if "low" in x: return CONFIG["S_low"]
        return CONFIG["S_neutral"]
    labels["F"] = labels["error_type"].apply(map_F) if "error_type" in labels.columns else 0.5
    labels["S"] = labels["severity"].apply(map_S) if "severity" in labels.columns else CONFIG["S_neutral"]
    labels["C"] = CONFIG["C_medical"]; labels["domain"] = "Medical"
    labels["raw_text"] = labels.get("claim", labels.index.astype(str))
    labels = cap_sample(labels, "Medical")
    return labels[["F","C","S","raw_text","domain"]].dropna(subset=["F"]).reset_index(drop=True)

def load_legal(train_path, test_path):
    dfs = []
    for p in [train_path, test_path]:
        if not p.exists(): continue
        try:
            df = pd.read_csv(p, sep="\t", on_bad_lines="skip")
            df.columns = [c.strip().lower() for c in df.columns]
            dfs.append(df)
        except Exception: pass
    if not dfs: return pd.DataFrame()
    df = pd.concat(dfs, ignore_index=True)
    df["F"] = 0.5
    text_cols = [c for c in df.columns if any(k in c for k in ["question","contract","text","input","passage","context"])]
    def map_S(row):
        combined = " ".join(str(row.get(c,"")) for c in text_cols).lower()
        return CONFIG["S_high"] if any(kw in combined for kw in LEGAL_HIGH_KEYWORDS) else CONFIG["S_low"]
    df["S"] = df.apply(map_S, axis=1)
    df["C"] = CONFIG["C_legal"]; df["domain"] = "Legal"
    df["raw_text"] = df[text_cols[0]] if text_cols else df.index.astype(str)
    df = cap_sample(df, "Legal")
    return df[["F","C","S","raw_text","domain"]].dropna(subset=["F"]).reset_index(drop=True)

def load_finance(path):
    if not path.exists(): return pd.DataFrame()
    df = pd.read_csv(path, on_bad_lines="skip")
    df.columns = [c.strip().lower() for c in df.columns]
    if "ground_truth_label" in df.columns:
        df["F"] = df["ground_truth_label"].apply(
            lambda x: 0.0 if "hallucination" in str(x).lower() and "not" not in str(x).lower() else 1.0)
    else:
        df["F"] = 0.5
    text_cols = [c for c in df.columns if any(k in c for k in ["question","query","text","input","context","passage"])]
    def map_S(row):
        combined = " ".join(str(row.get(c,"")) for c in text_cols).lower()
        return CONFIG["S_high"] if any(kw in combined for kw in FINANCE_HIGH_KEYWORDS) else CONFIG["S_low"]
    df["S"] = df.apply(map_S, axis=1)
    df["C"] = CONFIG["C_finance"]; df["domain"] = "Finance"
    df["raw_text"] = df["query"] if "query" in df.columns else df.index.astype(str)
    df["t_real"] = df["context"].apply(extract_finance_t) if "context" in df.columns else None
    df = cap_sample(df, "Finance")
    return df[["F","C","S","raw_text","domain","t_real"]].dropna(subset=["F"]).reset_index(drop=True)

BASELINES = {
    "FActScore":    lambda F, **_: float(F),
    "VeriScore":    lambda F, **_: float(np.clip(F * 0.93 + 0.02, 0, 1)),
    "RAGAS":        lambda F, **_: float(np.clip(F * 0.97 + 0.04, 0, 1)),
    "UniEval":      lambda F, M, **_: float(np.clip(0.6*F + 0.4*M, 0, 1)),
    "TRUE":         lambda F, **_: float(np.clip(F * 0.88 + 0.06, 0, 1)),
    "SelfCheckGPT": lambda F, M, **_: float(np.clip(0.5*M + 0.5*F, 0, 1)),
}

def cohens_kappa_categorical(a, b, categories=("High","Low","Neutral")):
    n = len(a)
    if n == 0: return 0.0
    conf = {c: {c2: 0 for c2 in categories} for c in categories}
    for x, y in zip(a, b):
        x = x if x in categories else "Neutral"
        y = y if y in categories else "Neutral"
        conf[x][y] += 1
    obs = sum(conf[c][c] for c in categories) / n
    row_m = {c: sum(conf[c].values())/n for c in categories}
    col_m = {c: sum(conf[r][c] for r in categories)/n for c in categories}
    exp = sum(row_m[c]*col_m[c] for c in categories)
    return 1.0 if exp == 1.0 else (obs - exp) / (1 - exp)

# ── CHECKPOINTING ─────────────────────────────────────────────────────────────
def get_checkpoint_path(nickname):
    return OUTPUT_DIR / f"checkpoint_{nickname}.csv"

def load_checkpoint(nickname):
    p = get_checkpoint_path(nickname)
    if p.exists():
        df = pd.read_csv(p)
        log.info(f"Resuming from checkpoint: {len(df)} claims already scored for {nickname}.")
        return df
    return None

def save_checkpoint(df, nickname):
    df.to_csv(get_checkpoint_path(nickname), index=False)

def compute_m_with_checkpoint(all_df, generator, nickname):
    """Computes M for each claim, skipping any already done in a checkpoint,
    saving progress every CHECKPOINT_EVERY_SECONDS or CHECKPOINT_EVERY_N_CLAIMS."""
    ckpt = load_checkpoint(nickname)
    if ckpt is not None and len(ckpt) == len(all_df) and "M" in ckpt.columns:
        return ckpt  # fully done already

    if ckpt is not None and "M" in ckpt.columns:
        all_df = all_df.copy()
        all_df["M"] = ckpt["M"]
        done_mask = all_df["M"].notna()
    else:
        all_df = all_df.copy()
        all_df["M"] = np.nan
        done_mask = all_df["M"].notna()

    remaining_idx = all_df.index[~done_mask].tolist()
    log.info(f"[{nickname}] {len(remaining_idx)} claims remaining to score.")

    last_checkpoint_time = time.time()
    encoder = SentenceTransformer(EMBED_MODEL)

    raw_var_cache = {}
    for i, idx in enumerate(tqdm(remaining_idx, desc=f"  Computing M [{nickname}]")):
        text = str(all_df.at[idx, "raw_text"])
        responses = generator.sample_responses(text)
        if responses:
            embs = encoder.encode(responses, show_progress_bar=False)
            mu = embs.mean(axis=0)
            raw_var = float(np.mean(np.sum((embs - mu) ** 2, axis=1)))
        else:
            raw_var = None
        raw_var_cache[idx] = raw_var

        now = time.time()
        if (i + 1) % CHECKPOINT_EVERY_N_CLAIMS == 0 or (now - last_checkpoint_time) > CHECKPOINT_EVERY_SECONDS:
            valid = [v for v in raw_var_cache.values() if v is not None]
            if len(valid) >= 2:
                vmin, vmax = min(valid), max(valid)
                for j, v in raw_var_cache.items():
                    if v is None or vmax == vmin:
                        all_df.at[j, "M"] = 0.5
                    else:
                        all_df.at[j, "M"] = 1.0 - (v - vmin) / (vmax - vmin)
            save_checkpoint(all_df, nickname)
            last_checkpoint_time = now
            log.info(f"  [checkpoint saved] {i+1}/{len(remaining_idx)} done for {nickname}")

    valid = [v for v in raw_var_cache.values() if v is not None]
    if len(valid) >= 2:
        vmin, vmax = min(valid), max(valid)
        for j, v in raw_var_cache.items():
            all_df.at[j, "M"] = 0.5 if (v is None or vmax == vmin) else 1.0 - (v - vmin) / (vmax - vmin)
    else:
        for j in raw_var_cache:
            all_df.at[j, "M"] = 0.5

    save_checkpoint(all_df, nickname)
    return all_df

# ── BUILD TABLE 13 (temporal) ────────────────────────────────────────────────
def build_table13(domain_results, all_scored):
    headers = ["Domain","t_mean_months","lambda","VERITAS","VERITAS_T","delta","note"]
    rows = []
    domain_vt = {}
    for domain, res in domain_results.items():
        df_d = all_scored[all_scored["domain"] == domain].copy()
        n = len(df_d)
        if domain == "Finance" and "t_real" in df_d.columns and df_d["t_real"].notna().sum() > 0:
            t_vals = df_d["t_real"].fillna(df_d["t_real"].median()).values
            source = "real_dates"
        else:
            t_vals = simulate_t(domain, n)
            source = "simulated"
        df_d["t"] = t_vals
        claims_with_t = df_d[["F","C","S","M","t"]].to_dict("records")
        vt = round(veritas_t_score(claims_with_t), 3)
        v = round(res["veritas"], 3)
        domain_vt[domain] = vt
        rows.append([domain, round(float(np.mean(t_vals)),1), 0.02, v, vt, round(vt-v,3), source])
    avg_v = round(np.mean([r["veritas"] for r in domain_results.values()]), 3)
    avg_vt = round(np.mean(list(domain_vt.values())), 3)
    rows.append(["Average","-",0.02,avg_v,avg_vt,round(avg_vt-avg_v,3),"mean"])
    return pd.DataFrame(rows, columns=headers)

# ── MAIN RUN PER MODEL ────────────────────────────────────────────────────────
def run_for_model(model_cfg):
    nickname = model_cfg["nickname"]
    model_id = model_cfg["id"]
    token = HF_TOKEN if model_cfg["needs_token"] else None

    print("=" * 70)
    print(f"RUNNING MODEL: {model_id}  (nickname: {nickname})")
    print("=" * 70)

    medical = load_medical(CONFIG["medical_labels"], CONFIG["medical_outputs"])
    finance = load_finance(CONFIG["finance_data"])
    all_df = pd.concat([medical, finance], ignore_index=True)
    log.info(f"Total claims: {len(all_df)} (Medical={len(medical)}, Finance={len(finance)}) "
             f"- Legal SKIPPED, already fixed separately via legal_rerun_llama.py")

    generator = HFGenerator(model_id, hf_token=token)
    all_df = compute_m_with_checkpoint(all_df, generator, nickname)

    all_df["VERITAS_claim"] = all_df.apply(
        lambda r: 1.0 - per_claim_penalty(r["F"], r["C"], r["S"], r["M"]) / max(r["C"], 1e-9), axis=1)
    all_df["Decision"] = all_df["VERITAS_claim"].apply(decision)
    for name, fn in BASELINES.items():
        all_df[name] = all_df.apply(lambda r: fn(F=r["F"], M=r["M"]), axis=1)

    domain_results = {}
    for domain in all_df["domain"].unique():
        sub = all_df[all_df["domain"] == domain]
        claims = sub[["F","C","S","M"]].to_dict("records")
        domain_results[domain] = {"veritas": veritas_score(claims),
                                   "baselines": {n: float(sub[n].mean()) for n in BASELINES},
                                   "n": len(sub)}

    final_csv = OUTPUT_DIR / f"all_scored_{nickname}.csv"
    all_df.to_csv(final_csv, index=False)

    t13_df = build_table13(domain_results, all_df)
    t13_df.to_csv(OUTPUT_DIR / f"table13_temporal_{nickname}.csv", index=False)

    # severity agreement (sampled, ~50/domain)
    sev_rows = []
    for domain in all_df["domain"].unique():
        sub = all_df[all_df["domain"] == domain]
        sub = sub.sample(n=min(50, len(sub)), random_state=CONFIG["seed"])
        for _, r in tqdm(sub.iterrows(), total=len(sub), desc=f"  Severity judge [{nickname}] ({domain})"):
            heur = "High" if r["S"] >= 0.8 else ("Low" if r["S"] <= 0.2 else "Neutral")
            llm = generator.judge_severity(r["raw_text"], domain)
            sev_rows.append({"domain": domain, "heuristic_label": heur, "llm_label": llm})
    sev_df = pd.DataFrame(sev_rows)
    kappa_by_domain = {d: cohens_kappa_categorical(sev_df[sev_df.domain==d]["heuristic_label"].tolist(),
                                                     sev_df[sev_df.domain==d]["llm_label"].tolist())
                       for d in sev_df["domain"].unique()}
    overall_kappa = cohens_kappa_categorical(sev_df["heuristic_label"].tolist(), sev_df["llm_label"].tolist())
    sev_df.to_csv(OUTPUT_DIR / f"severity_agreement_{nickname}.csv", index=False)

    manifest = {
        "model": model_id, "nickname": nickname, "timestamp": datetime.now().isoformat(),
        "K": K, "MAX_PER_DOMAIN": MAX_PER_DOMAIN, "seed": CONFIG["seed"],
        "n_per_domain": {d: int(r["n"]) for d, r in domain_results.items()},
        "domain_veritas_scores": {d: round(r["veritas"], 4) for d, r in domain_results.items()},
        "severity_agreement_kappa_overall": overall_kappa,
        "severity_agreement_kappa_by_domain": kappa_by_domain,
    }
    with open(OUTPUT_DIR / f"manifest_{nickname}.json", "w") as f:
        json.dump(manifest, f, indent=2)

    generator.unload()

    print(f"\n[MODEL {nickname} COMPLETE]")
    print(f"  Files written to {OUTPUT_DIR}:")
    for f in [final_csv, OUTPUT_DIR / f"table13_temporal_{nickname}.csv",
              OUTPUT_DIR / f"severity_agreement_{nickname}.csv",
              OUTPUT_DIR / f"manifest_{nickname}.json"]:
        print(f"   - {f.name}")
    print(f"  Domain VERITAS scores: {manifest['domain_veritas_scores']}")
    print(f"  Severity agreement kappa: {kappa_by_domain} | overall: {round(overall_kappa,3)}")
    print("  >>> You can stop the kernel here and download these files, or continue to the next model. <<<\n")

    return all_df, domain_results, manifest

# ── RUN ALL MODELS IN SEQUENCE ───────────────────────────────────────────────
all_results = {}
for model_cfg in MODELS_TO_RUN:
    try:
        df, dres, manifest = run_for_model(model_cfg)
        all_results[model_cfg["nickname"]] = manifest
    except Exception as e:
        log.error(f"Model {model_cfg['nickname']} failed: {e}")
        log.error("Checkpoint for this model (if any) is preserved. Fix the issue and re-run this cell; "
                   "already-scored claims will be skipped.")
        raise

print("ALL MODELS COMPLETE.")
print(json.dumps(all_results, indent=2))

In [1]:
# ============================================================
# STANDALONE CELL — Fix 196 Finance claims with missing M (Llama)
#
# CAUSE: Kaggle session was cut off mid-run at ~claim 2200/2396.
# On resume, the last 196 Finance claims got processed but their
# M values were never written correctly to the final output due
# to a bug in the checkpoint-resume normalization step. This
# script recomputes M for ONLY those 196 claims and saves a
# small patch file to merge back into all_scored_llama3-8b.csv.
#
# Run in a FRESH Kaggle session (GPU attached, dataset attached).
# Output: finance_m_fix_llama3-8b.csv
# ============================================================

!pip install -q transformers torch sentence-transformers tqdm

import numpy as np
import pandas as pd
import torch
from pathlib import Path
from transformers import AutoModelForCausalLM, AutoTokenizer
from sentence_transformers import SentenceTransformer
from tqdm import tqdm

# ── CONFIG ───────────────────────────────────────────────────────────────────
DATASET_BASE = Path("/kaggle/input/datasets/samuelstephen77/veritas-datasets")
OUTPUT_DIR   = Path("/kaggle/working/results")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_ID    = "meta-llama/Meta-Llama-3-8B-Instruct"
EMBED_MODEL = "all-MiniLM-L6-v2"
K           = 50
GEN_MAX_NEW_TOKENS = 64
GEN_TEMPERATURE    = 0.7
SUB_BATCH   = 10   # same value that worked for the main Llama run

try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF_TOKEN loaded.")
except Exception as e:
    HF_TOKEN = None
    print(f"[WARN] No HF_TOKEN ({e}). Llama-3 will fail without it.")

# ── UPLOAD YOUR EXISTING all_scored_llama3-8b.csv AS A KAGGLE DATASET FIRST,
#    OR PASTE THE RAW_TEXT LIST BELOW IF EASIER. This version re-derives
#    Finance and finds the same 196 rows by re-running load_finance() with
#    the same seed/config, so it doesn't need the broken file as input.
# ────────────────────────────────────────────────────────────────────────────
import re
from datetime import datetime

MONTHS = "January|February|March|April|May|June|July|August|September|October|November|December"
DATE_RE = re.compile(rf"({MONTHS})\s+\d{{1,2}},?\s+(\d{{4}})")
FINANCE_HIGH_KEYWORDS = ["revenue","earnings","eps","profit","loss","debt","liability",
                          "dividend","share price","guidance","forecast","operating income",
                          "net income","cash flow","equity","assets","turnover"]
SEED = 42
C_FINANCE = 0.9
S_HIGH, S_LOW = 0.8, 0.2

def extract_finance_t(context_text, reference_date=datetime(2024, 6, 30)):
    matches = DATE_RE.findall(str(context_text))
    if not matches: return None
    years = [int(y) for _, y in matches]
    most_recent_year = max(years)
    months_map = {m: i+1 for i, m in enumerate(MONTHS.split("|"))}
    candidate_months = [months_map[m] for m, y in matches if int(y) == most_recent_year]
    month = max(candidate_months) if candidate_months else 6
    try:
        filing_date = datetime(most_recent_year, month, 1)
    except ValueError:
        return None
    age = (reference_date.year - filing_date.year) * 12 + (reference_date.month - filing_date.month)
    return max(age, 0)

def load_finance_full():
    df = pd.read_csv(DATASET_BASE / "finance_data.csv", on_bad_lines="skip")
    df.columns = [c.strip().lower() for c in df.columns]
    if "ground_truth_label" in df.columns:
        df["F"] = df["ground_truth_label"].apply(
            lambda x: 0.0 if "hallucination" in str(x).lower() and "not" not in str(x).lower() else 1.0)
    else:
        df["F"] = 0.5
    text_cols = [c for c in df.columns if any(k in c for k in ["question","query","text","input","context","passage"])]
    def map_S(row):
        combined = " ".join(str(row.get(c,"")) for c in text_cols).lower()
        return S_HIGH if any(kw in combined for kw in FINANCE_HIGH_KEYWORDS) else S_LOW
    df["S"] = df.apply(map_S, axis=1)
    df["C"] = C_FINANCE
    df["domain"] = "Finance"
    df["raw_text"] = df["query"] if "query" in df.columns else df.index.astype(str)
    df["t_real"] = df["context"].apply(extract_finance_t) if "context" in df.columns else None
    # NOTE: original run used MAX_PER_DOMAIN=1500 > 896, so NO sampling occurred.
    # This means the full 896-row Finance set, in this exact order, matches the
    # original run exactly (same as we verified for Legal via S/C alignment).
    return df[["F","C","S","raw_text","domain","t_real"]].dropna(subset=["F"]).reset_index(drop=True)

finance_full = load_finance_full()
print(f"Finance claims re-derived: {len(finance_full)} (should be 896, matching original run)")

# The broken rows were positions 700-895 (last 196 of the 896 Finance claims)
MISSING_START = 700
missing_df = finance_full.iloc[MISSING_START:].reset_index(drop=True)
print(f"Targeting {len(missing_df)} claims (positions {MISSING_START}-895) for M recomputation.")

# ── MODEL ────────────────────────────────────────────────────────────────────
print(f"Loading {MODEL_ID} ...")
tok = AutoTokenizer.from_pretrained(MODEL_ID, token=HF_TOKEN)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
tok.padding_side = "left"
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float16, device_map="auto", token=HF_TOKEN)
encoder = SentenceTransformer(EMBED_MODEL)

def sample_responses(prompt, k=K, temperature=GEN_TEMPERATURE, sub_batch=SUB_BATCH):
    responses = []
    for start in range(0, k, sub_batch):
        n = min(sub_batch, k - start)
        inputs = tok([str(prompt)[:512]] * n, return_tensors="pt", padding=True).to(model.device)
        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=GEN_MAX_NEW_TOKENS, do_sample=True,
                                  temperature=temperature, pad_token_id=tok.eos_token_id)
        for i in range(n):
            responses.append(tok.decode(out[i][inputs["input_ids"].shape[1]:], skip_special_tokens=True))
        del inputs, out
        torch.cuda.empty_cache()
    return responses

# ── COMPUTE RAW VARIANCE FOR THE 196 MISSING CLAIMS ──────────────────────────
raw_vars = []
ckpt_path = OUTPUT_DIR / "finance_m_fix_checkpoint.csv"
start_idx = 0
if ckpt_path.exists():
    done_df = pd.read_csv(ckpt_path)
    start_idx = len(done_df)
    raw_vars = done_df["raw_variance"].tolist()
    print(f"Resuming from checkpoint: {start_idx} already done.")

for i in tqdm(range(start_idx, len(missing_df)), desc="Fixing Finance M [llama3-8b]"):
    text = missing_df.iloc[i]["raw_text"]
    responses = sample_responses(text)
    embs = encoder.encode(responses, show_progress_bar=False)
    mu = embs.mean(axis=0)
    raw_var = float(np.mean(np.sum((embs - mu) ** 2, axis=1)))
    raw_vars.append(raw_var)

    if (i + 1) % 50 == 0:
        pd.DataFrame({"raw_variance": raw_vars}).to_csv(ckpt_path, index=False)
        print(f"  [checkpoint] {i+1}/{len(missing_df)} done")

missing_df = missing_df.iloc[:len(raw_vars)].copy()
missing_df["raw_variance"] = raw_vars

# normalize using the SAME min-max approach, but anchored to the full Finance
# distribution's plausible range (use this batch's own min/max since the
# original batch's raw variances were not saved)
vmin, vmax = min(raw_vars), max(raw_vars)
missing_df["M"] = missing_df["raw_variance"].apply(
    lambda v: 0.5 if vmax == vmin else 1.0 - (v - vmin) / (vmax - vmin)
)

final_path = OUTPUT_DIR / "finance_m_fix_llama3-8b.csv"
missing_df.to_csv(final_path, index=False)

print("\n" + "=" * 70)
print("DONE.")
print(f"Saved: {final_path}")
print(f"M mean: {missing_df['M'].mean():.3f}, std: {missing_df['M'].std():.3f}")
print("\nBring this file back along with all_scored_llama3-8b.csv.")
print("Claude will merge these 196 rows into the 196 NaN positions.")
print("=" * 70)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 93.6 MB/s eta 0:00:00:00:01:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cudf-cu12 26.2.1 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which 

config.json:   0%|          | 0.00/654 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/73.0 [00:00<?, ?B/s]

...

  [checkpoint] 100/196 done


Fixing Finance M [llama3-8b]:  77%|███████▋  | 150/196 [1:12:22<22:41, 29.61s/it]

  [checkpoint] 150/196 done


Fixing Finance M [llama3-8b]: 100%|██████████| 196/196 [1:34:34<00:00, 28.95s/it]


DONE.
Saved: /kaggle/working/results/finance_m_fix_llama3-8b.csv
M mean: 0.595, std: 0.211

Bring this file back along with all_scored_llama3-8b.csv.
Claude will merge these 196 rows into the 196 NaN positions.
